# N-Body Gravity Simulation

Gravitational N-body simulation powered by the **Minkowski ECS** engine.

Bodies interact via Newtonian gravity with softening to prevent singularities.
The toroidal world wraps at boundaries.

## Setup

```bash
cd crates/minkowski-py
maturin develop --release
```

In [ ]:
import minkowski_py as mk
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

## 1. Run the simulation

500 bodies, 500 timesteps. The O(N^2) direct summation is fine at this scale.

In [ ]:
sim = mk.NBodySim(
    n=500,
    world_size=500.0,
    g=0.06674,
    softening=1.0,
    dt=0.001,
)

sim.step(500, record=True)
print(f"Simulated {sim.current_frame()} frames, {sim.entity_count()} bodies")

## 2. Final state

In [ ]:
df = sim.to_polars()
df.head(10)

## 3. Position scatter — initial vs final

In [ ]:
history = sim.history_to_polars()

first_frame = history.filter(pl.col("frame") == 0)
last_frame = history.filter(pl.col("frame") == history["frame"].max())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, frame_df, title in [
    (axes[0], first_frame, "Initial positions"),
    (axes[1], last_frame, f"After {sim.current_frame()} steps"),
]:
    ax.scatter(
        frame_df["pos_x"].to_numpy(),
        frame_df["pos_y"].to_numpy(),
        s=frame_df["mass"].to_numpy() * 10,
        alpha=0.6,
        c="steelblue",
        edgecolors="navy",
        linewidths=0.3,
    )
    ax.set_xlim(0, 500)
    ax.set_ylim(0, 500)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.grid(True, alpha=0.2)

plt.suptitle("N-Body Gravitational Simulation", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Trajectory traces

Plot the paths of a random sample of bodies over time.

In [ ]:
# Sample 20 bodies by index within each frame
n_bodies = first_frame.shape[0]
sample_indices = np.random.choice(n_bodies, size=min(20, n_bodies), replace=False)

fig, ax = plt.subplots(figsize=(8, 8))

# Group history by body index within each frame
history_np = history.select(["frame", "pos_x", "pos_y"]).to_numpy()
n_frames = int(history["frame"].max()) + 1

for body_idx in sample_indices:
    # Extract this body's trajectory (every n_bodies-th row, offset by body_idx)
    body_rows = history_np[body_idx::n_bodies]
    ax.plot(body_rows[:, 1], body_rows[:, 2], alpha=0.4, linewidth=0.5)

ax.set_xlim(0, 500)
ax.set_ylim(0, 500)
ax.set_aspect("equal")
ax.set_title(f"Trajectories of {len(sample_indices)} sampled bodies")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Kinetic energy over time

In [ ]:
# Rerun with velocity tracking
sim2 = mk.NBodySim(n=300, g=0.1, softening=2.0, dt=0.002)
sim2.step(300, record=True)

# NBody history doesn't include velocity, so we compute displacement-based KE proxy
# For a proper analysis, use to_polars() at each step
df_final = sim2.to_polars()
speeds = np.sqrt(df_final["vel_x"].to_numpy()**2 + df_final["vel_y"].to_numpy()**2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(speeds, bins=40, color="steelblue", edgecolor="navy", alpha=0.7)
ax.set_xlabel("Speed")
ax.set_ylabel("Count")
ax.set_title(f"Velocity distribution after {sim2.current_frame()} steps")
ax.axvline(np.mean(speeds), color="red", linestyle="--", label=f"mean={np.mean(speeds):.2f}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Gravitational constant sweep

In [ ]:
g_values = [0.01, 0.05, 0.1, 0.5]

fig, axes = plt.subplots(1, len(g_values), figsize=(16, 4))

for ax, g in zip(axes, g_values):
    s = mk.NBodySim(n=300, g=g, softening=2.0, dt=0.002)
    s.step(200)
    df = s.to_polars()
    ax.scatter(
        df["pos_x"].to_numpy(),
        df["pos_y"].to_numpy(),
        s=3, alpha=0.6, c="steelblue",
    )
    ax.set_xlim(0, 500)
    ax.set_ylim(0, 500)
    ax.set_aspect("equal")
    ax.set_title(f"G = {g}")

plt.suptitle("Effect of gravitational constant on clustering", fontsize=14)
plt.tight_layout()
plt.show()